# Reproducing Gonzalez & Giocomo (2024) from DANDI:000405

Paper: Gonzalez A, Giocomo LM (2024). Parahippocampal neurons encode task-relevant information for
goal-directed navigation. eLife 12:RP85646, doi:10.7554/elife.85646.

Dataset: DANDI:000405, version `draft`. The dandiset has no published version (the archive lists
only the draft, last modified 2023-06-20T00:56:24Z), so the draft is the only version that can be
read; the notebook checks that its modification time is still that one. The paper cites no version.

Claims: `claims/gonzalez-2024.yaml`, written from the eLife reviewed preprint as revised (Europe PMC,
PMC10942598), with the Methods, all figure supplements, and the paper's code
(github.com/alexgonzl/TMA at commit 6f93593).

What the files hold, found below: for each Tree-Maze session, firing rates of every unit in 20 ms
bins (smoothed, not spikes), tracked position and head angle, a trial table and an event table; for
each open-field session, firing rates and tracking. There are no spike times and no waveforms.

| claim | tested here |
|---|---|
| fig1d-task-performance | tested |
| fig1-switch-performance | tested |
| fig1-training-to-criterion | blocked, data_missing (training sessions not in the dandiset) |
| fig2c-cue-error-coding | tested |
| fig2d-cue-remapping | tested |
| fig2e-unit-remap-behaviour | tested |
| fig2f-session-remap-behaviour | tested |
| fig2g-population-remap-behaviour | tested |
| fig2h-subject-remap-behaviour | tested |
| fig2-remap-behaviour-controls | tested |
| fig3d-cue-rate-remapping-model | tested |
| fig3e-zone-decoder-cue | blocked, other (the decoders were not run in this attempt; the data are in the dandiset) |
| fig3f-decision-decoder | blocked, other (the decoders were not run in this attempt; the data are in the dandiset) |
| fig3g-decoder-tracks-behaviour | blocked, other (the decoders were not run in this attempt; the data are in the dandiset) |
| fig4c-reward-error-coding | tested |
| fig4d-reward-remapping | tested |
| fig4ef-reward-remap-behaviour | tested |
| fig4gh-reward-remap-population-subject | tested |
| fig4-reward-remap-behaviour-controls | tested |
| fig5-reward-rate-remapping-model | tested |
| fig4s5-cue-reward-remap-correlated | tested |
| fig4s6-incorrect-coding-overlap | tested |
| fig4s7-error-signal-persists | tested |
| fig6cd-of-aggregate-model | tested |
| fig7b-matched-units | blocked, data_missing (no waveforms or spikes) |
| fig7de-hd-speed-tradeoff | blocked, other (not tested; the clustering of the paper's matched units was not run) |
| fig7f-tuning-metrics-match | blocked, other (not tested; depends on fig7de-hd-speed-tradeoff) |
| fig7g-hd-cluster-remaps-more | blocked, other, in the verified record; tested here on the paper's clusters for a proposed change |
| fig7h-hd-cluster-encoder-scores | blocked, other, in the verified record; tested here on the paper's clusters for a proposed change |
| methods-lookahead-encoder-best | tested |

In [ ]:
import os, time, zlib, warnings, resource
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import tma

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
PAPER = "gonzalez-2024"


def seed_of(claim):
    """The claim's seed (docs/protocol.md): crc32 of '<paper key>/<claim id>'."""
    return zlib.crc32(f"{PAPER}/{claim}".encode())


T_START = time.time()
info = tma.version_info()
print("dandiset", tma.DANDISET, "version", info["version"], "modified", info["modified"],
      "status", info["status"], "assets", info["asset_count"], "bytes", info["size"])
assert info["modified"] == tma.DRAFT_MODIFIED, "the draft has changed since this notebook was written"
assets = tma.list_assets()
print(len(assets), "assets")

## Data check

Every file's metadata and tables are read (no large arrays), then the Tree-Maze sessions' rates and
tracking are streamed.

In [ ]:
meta = tma.read_all_meta(assets)
rows = []
for r in meta:
    u = r["units"]
    rows.append(dict(path=r["path"], session=r["session"], task=r["task"], subject_id=r["subject_id"],
                     subject=tma.SUBJECTS[r["subject_id"]], n_units=len(u),
                     n_cell=int((u.unit_type == "cell").sum()) if len(u) else 0,
                     n_mua=int((u.unit_type == "mua").sum()) if len(u) else 0,
                     n_samps=r["fr_shape"][0], fr_cols=r["fr_shape"][1], td_rows=r["td_shape"][0],
                     td_cols=r["td_shape"][1], td_desc=r["td_desc"], fr_unit=r["fr_unit"],
                     fr_rate=r["fr_rate"], td_rate=r["td_rate"], td_unit=r["td_unit"],
                     n_trials=len(r["trials"]) if "trials" in r else np.nan,
                     acquisition=",".join(r["acquisition"])))
files = pd.DataFrame(rows)
files["tm"] = files.task != "OF"
meta_by_session = {r["session"]: r for r in meta}

In [ ]:
print("files by subject and task")
print(pd.crosstab(files.subject, files.task, margins=True))
print("\nacquisition contents:\n", files.groupby("task").acquisition.value_counts().to_string())
print("\ntrack_data description by task:\n", files.groupby("task").td_desc.value_counts().to_string())
print("\nfiring_rates unit:", files.fr_unit.unique(), " rate (Hz):", files.fr_rate.unique(),
      " track_data rate attribute:", files.td_rate.unique(), " track unit:", files.td_unit.unique())
print("session_start_time years:", sorted({r["start"][:4] for r in meta}), " example:", meta[0]["start"])
print("units per firing_rates column count equal to unit_table rows:", (files.fr_cols == files.n_units).all())
print("track rows = rate samples - 1, fraction of files by task:",
      files.assign(one_less=files.td_rows == files.n_samps - 1).groupby("task").one_less.mean().to_dict())
print("track columns by task:", files.groupby("task").td_cols.unique().to_dict())
tmf = files[files.tm]
print(f"\nTree-Maze files {len(tmf)}: units {tmf.n_units.sum()} (cell {tmf.n_cell.sum()}, mua {tmf.n_mua.sum()})")
print(f"open-field files {(~files.tm).sum()}: units {files[~files.tm].n_units.sum()}")
print("\nunit_type values:", sorted(set(np.concatenate([r["units"].unit_type.values for r in meta if len(r["units"])]))))
ex = meta_by_session["Li_T3g_062518"]
print("\ntrial_table columns:", list(ex["trials"].columns))
print("event_table columns:", list(ex["events"].columns))
print("unit_table columns:", list(ex["units"].columns))
print("event names:", sorted(set(np.concatenate([r["events"].event.unique() for r in meta if "events" in r]))))
alltr = pd.concat([r["trials"].assign(session=r["session"]) for r in meta if "trials" in r])
print("\ntrial values: cue", alltr.cue.value_counts().to_dict(), " dec", alltr.dec.value_counts().to_dict(),
      " correct", alltr.correct.value_counts(dropna=False).to_dict(), " sw", alltr.sw.value_counts().to_dict())
print("trials with tE_2 < t0:", int((alltr.tE_2 < alltr.t0).sum()), " tR < tE_2:", int((alltr.tR < alltr.tE_2).sum()))
print("duplicated uuids across files:", int(pd.concat([r["units"].uuid for r in meta]).duplicated().sum()))
dup_n = files[files.duplicated("n_samps", keep=False)][["session", "n_samps", "n_units"]]
print("files sharing a sample count:\n", dup_n.to_string())

The paper's code (experiment_info.py, `session_to_nwb`) writes `track_data` as the session's track table
`get_track_data()[1:]`, whose columns are t, x, y, ha; the file's description says "columns=x,y,hd". The
checks below show that the first column is a clock in seconds and the head angle is in radians, and the
`rate` attribute of `track_data` is 0.02, the sample period, not a rate. Rates are labelled "uv" but are
firing rates in spikes per second (their range below). Every `session_start_time` is in year 0018 or 0019:
the session dates (2018 and 2019 in the session names) were written with a two-digit year. Two files share a sample count; they are from
different subjects and tasks with different unit counts, so they are not one recording deposited twice.
`Cl_T3g_121418` and `Cl_T3g_121418_0` are two recordings on the same day with different trial sequences;
the paper's code leaves `Cl_T3g_121418_0` out (`SummaryInfo.invalid_sessions`), and so does this notebook.

In [ ]:
# The repository's metadata tables, from the paper's code at the cited commit (an input), to compare
# the session list with the dandiset.
TMA = "https://raw.githubusercontent.com/alexgonzl/TMA/6f935935d93a9a27f931f6e73c05d965dfbb862a/metadata/"
repo_sessions = pd.read_csv(TMA + "session_table.csv", index_col=0)
repo_perf = pd.read_csv(TMA + "perf_table.csv")
norm = lambda s: s.lower()
in_ds = set(files.session.map(norm))
missing = repo_sessions[~repo_sessions.session.map(norm).isin(in_ds)]
print(f"repository session table: {len(repo_sessions)} sessions; in the dandiset: {len(repo_sessions) - len(missing)}")
print("sessions in the repository's session table and not in the dandiset:")
print(missing[["session", "task_orig", "n_cells", "n_mua"]].to_string())
print("units in those sessions:", int(missing.n_cells.sum() + missing.n_mua.sum()),
      "(cells", int(missing.n_cells.sum()), ")")
print("dandiset sessions not in the repository table:",
      sorted(set(files.session) - set(repo_sessions.session)))
m = files.assign(k=files.session.map(norm)).merge(repo_perf.assign(k=repo_perf.session.map(norm)), on="k")
print("Tree-Maze sessions in both:", len(m), "; trial counts differ:", int((m.n_trials_x != m.n_trials_y).sum()),
      "; unit counts differ:", int((m.n_units_x != m.n_units_y).sum()))

### Loading the Tree-Maze sessions

Every Tree-Maze file is streamed once. For each, the loader keeps the trial zone rates (39 zones,
outbound and inbound) under the code's default blanking and under the two control blankings
(reward, speed), the speed and the sample-level data the checks below need, and drops the rest.

In [ ]:
# Zone encoders (claims fig3d, fig5 and methods-lookahead-encoder-best): models as in
# zone_encoder_comps_dict ('lag', 'cue', 'rw'). Folds: 5 (Methods) under the claim's seed and four more,
# and 10 (the code's default) under the claim's seed.
LAG_MODELS = {"Z-50": (-50, None), "Z0": (0, None), "Z50": (50, None)}
CUE_MODELS = {"Z50": (50, None), "Z+C": (50, "fixed"), "ZxC": (50, "inter")}
RW_MODELS = {"Zi": (50, None), "Zi+R": (50, "fixed"), "ZixR": (50, "inter")}


def enc_runs(claim):
    return [(5, seed_of(claim) + o, f"5 folds, seed+{o}") for o in [0, 1000, 2000, 3000, 4000]] + \
           [(10, seed_of(claim), "10 folds")]


def load_tm(row):
    a = assets[assets.path == row.path].iloc[0]
    fr, track = tma.read_session(a.asset_id)
    r = meta_by_session[row.session]
    out = dict(session=row.session, subject=row.subject, task=row.task, units=r["units"],
               trials=r["trials"], events=r["events"])
    out["fr_range"] = (float(np.nanmin(fr)), float(np.nanmax(fr)), float(np.nanmean(fr)))
    out["track_range"] = track.describe().loc[["min", "max"]]
    out["dt"] = np.nanmedian(np.diff(track.t.values))
    out["ses"] = tma.Session(fr, track, r["trials"], r["events"])
    if len(r["trials"]) >= tma.MIN_TRIALS and fr.shape[0] >= tma.MIN_UNITS:
        out["ses_rw"] = tma.Session(fr, track, r["trials"], r["events"], reward_blank=True)
        out["ses_sp"] = tma.Session(fr, track, r["trials"], r["events"], speed_blank=True)
        # positions of the trial markers, for the clock checks
        pz = out["ses"].pz
        out["zone_at"] = {k: pz[np.clip(r["trials"][k].values.astype(int), 0, len(pz) - 1)]
                          for k in ["t0", "tD", "tE_2", "tR"]}
        # clock check: even against odd outbound zone maps with rates shifted against position
        out["clock"] = tma.clock_check(fr, out["ses"], shifts=(0, 250, -250))
        if row.session not in tma.INVALID_SESSIONS:
            out["enc_cue"] = tma.encoder_scores(fr, out["ses"], "out", CUE_MODELS, enc_runs("fig3d-cue-rate-remapping-model"))
            out["enc_lag"] = tma.encoder_scores(fr, out["ses"], "out", LAG_MODELS, enc_runs("methods-lookahead-encoder-best"))
            out["enc_rw"] = tma.encoder_scores(fr, out["ses"], "in", RW_MODELS, enc_runs("fig5-reward-rate-remapping-model"))
    return out


from concurrent.futures import ThreadPoolExecutor
t0 = time.time()
tm_rows = [row for _, row in files[files.tm].iterrows() if row.n_units > 0]
with ThreadPoolExecutor(6) as ex:
    TM = {d["session"]: d for d in ex.map(load_tm, tm_rows)}
print(f"loaded {len(TM)} Tree-Maze sessions with units in {time.time() - t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

In [ ]:
rng_rows = []
for s, d in TM.items():
    tr = d["track_range"]
    rng_rows.append(dict(session=s, subject=d["subject"], n_units=len(d["units"]), n_trials=len(d["trials"]),
                         fr_min=d["fr_range"][0], fr_max=d["fr_range"][1], fr_mean=d["fr_range"][2],
                         t_min=tr.loc["min", "t"], x_min=tr.loc["min", "x"], x_max=tr.loc["max", "x"],
                         y_min=tr.loc["min", "y"], y_max=tr.loc["max", "y"], ha_min=tr.loc["min", "ha"],
                         ha_max=tr.loc["max", "ha"], dt=d["dt"], blanked=d["ses"].n_blank / d["ses"].n_samps))
R = pd.DataFrame(rng_rows)
print("ranges over Tree-Maze sessions with units:")
print(R.describe().loc[["min", "50%", "max"]].round(3).to_string())
print("\nfirst track column steps by", R.dt.min().round(5), "to", R.dt.max().round(5),
      "s per sample: a clock in seconds at 50 Hz; x and y within the maze limits in mm "
      "(maze polygons span x -800 to 800, y -80 to 1450); head angle 0 to 2 pi, radians.")
print("firing rates are non-negative, up to", R.fr_max.max().round(1), "per second; mean of session means",
      R.fr_mean.mean().round(2))
print("fraction of samples blanked (outside zones or invalid transitions): median",
      R.blanked.median().round(3), "max", R.blanked.max().round(3))
bad = alltr[alltr.tE_2 < alltr.t0]
print("\ntrials whose start is after their end (left as they are; such a trial has no outbound samples):")
print(bad[["session", "t0", "tE", "tE_2", "tR", "dur", "correct"]].to_string())

### Clocks

Rates, tracking and trial markers share one sample index. Two checks: (1) where the tracked position is
at each trial marker (a contrast: the zone at the end of the outbound run against the recorded decision);
(2) per unit, the agreement of the outbound zone maps of even and odd trials, with the rates as they are
and shifted by 5 s against the position (a contrast between true and shifted alignment).

In [ ]:
names = np.array(tma.TMZ.all_segs_names + ["none"])
left_goals, right_goals = {"G3", "G4"}, {"G1", "G2"}
ck = []
for s, d in TM.items():
    if "zone_at" not in d:
        continue
    z = {k: names[v] for k, v in d["zone_at"].items()}
    t = d["trials"]
    end_left = np.isin(z["tE_2"], list(left_goals))
    end_right = np.isin(z["tE_2"], list(right_goals))
    ck.append(dict(session=s, subject=d["subject"], n=len(t),
                   t0_home=np.mean(z["t0"] == "H"), tD_dec=np.mean(z["tD"] == "D"), tR_home=np.mean(z["tR"] == "H"),
                   dec_match=np.mean((end_left & (t.dec == "L").values) | (end_right & (t.dec == "R").values)),
                   dec_match_shuffled=np.mean((end_left & (np.roll(t.dec.values, 1) == "L")) |
                                              (end_right & (np.roll(t.dec.values, 1) == "R"))),
                   map_agree=d["clock"][0], map_agree_shift_plus5s=d["clock"][250],
                   map_agree_shift_minus5s=d["clock"][-250]))
CK = pd.DataFrame(ck)
print(CK.drop(columns=["session"]).groupby("subject").median().round(3).to_string())
print("\nall sessions, median:", CK.drop(columns=["session", "subject"]).median().round(3).to_dict())
CK["aligned_best"] = CK.map_agree > CK[["map_agree_shift_plus5s", "map_agree_shift_minus5s"]].max(axis=1)
print("sessions where unshifted maps agree best:", int(CK.aligned_best.sum()), "of", int(CK.map_agree.notna().sum()))
print("sessions where it could not be shown:", CK.loc[~CK.aligned_best, "session"].tolist())

## Behaviour per session

Per session, from the trial table (the code's `BehaviorData.get_session_perf`): trials, Switch trials,
fraction correct over the trials that have a correct flag. Included sessions: at least one unit and at
least 50 trials (Results, Figure 1d).

In [ ]:
beh = []
for s, r in meta_by_session.items():
    if "trials" not in r:
        continue
    p = tma.session_perf(r["trials"])
    p.update(session=s, subject=tma.SUBJECTS[r["subject_id"]], task=s.split("_")[1], n_units=len(r["units"]),
             n_cells=int((r["units"].unit_type == "cell").sum()) if len(r["units"]) else 0)
    beh.append(p)
BEH = pd.DataFrame(beh).set_index("session")
incl = BEH[(BEH.n_units >= tma.MIN_UNITS) & (BEH.n_trials >= tma.MIN_TRIALS)]
print(f"Tree-Maze sessions {len(BEH)}; included {len(incl)}; units in them {incl.n_units.sum()} "
      f"(cells {incl.n_cells.sum()})")
print(incl.groupby("subject").agg(sessions=("n_trials", "size"), units=("n_units", "sum"), cells=("n_cells", "sum"),
                                  median_trials=("n_trials", "median"), median_acc=("pct_correct", "median")).to_string())

## fig1d-task-performance

> Across subjects, the median session accuracy on the Tree-Maze was 72.6% and the median number of
> trials per session was 110, among sessions with at least one unit and at least 50 trials.

In [ ]:
subj_med = incl.groupby("subject")[["pct_correct", "n_trials", "pct_sw_correct", "n_sw_trials"]].median()
print(subj_med.round(3).to_string())
acc_ms, ntr_ms = subj_med.pct_correct.median() * 100, subj_med.n_trials.median()
acc_pool, ntr_pool = incl.pct_correct.median() * 100, incl.n_trials.median()
print(f"\nmedian of subject medians: accuracy {acc_ms:.1f}% (expected 72.6 +- 7.26), trials {ntr_ms:.1f} (expected 110 +- 11)")
print(f"variant, pooled sessions: accuracy {acc_pool:.1f}%, trials {ntr_pool:.1f}")
for lo in [40, 50, 60]:
    sub = BEH[(BEH.n_units >= 1) & (BEH.n_trials >= lo)]
    sm = sub.groupby("subject")[["pct_correct", "n_trials"]].median().median()
    print(f"  threshold {lo} trials: {len(sub)} sessions, accuracy {sm.pct_correct * 100:.1f}%, trials {sm.n_trials:.1f}")
print("repository perf_table.csv, same rule:",
      repo_perf[(repo_perf.n_units >= 1) & (repo_perf.n_trials >= 50)].groupby("subject")[["pct_correct", "n_trials", "pct_sw_correct", "n_sw_trials"]]
      .median().median().round(3).to_dict())
# Trials per session per subject under other selections, to look for one that gives the paper's 110.
SUBJ = sorted(BEH.subject.unique())
sel = {"all Tree-Maze sessions": BEH,
       ">= 1 unit, >= 50 trials (the rule)": incl,
       ">= 1 isolated unit, >= 50 trials": incl[incl.n_cells >= 1],
       ">= 50 trials, T3g sessions only": incl[incl.task == "T3g"],
       ">= 50 trials, invalid session left out": incl.drop(index=[s for s in tma.INVALID_SESSIONS if s in incl.index]),
       ">= 1 unit, >= 100 trials": BEH[(BEH.n_units >= 1) & (BEH.n_trials >= 100)]}
tab = pd.DataFrame({k: v.groupby("subject").n_trials.median() for k, v in sel.items()}).T[SUBJ]
tab.insert(0, "sessions", [len(v) for v in sel.values()])
tab["median of subject medians"] = tab[SUBJ].median(axis=1)
tab["mean of subject medians"] = tab[SUBJ].mean(axis=1)
tab["pooled median"] = [v.n_trials.median() for v in sel.values()]
tab["Switch trials, median of subject medians"] = [v.groupby("subject").n_sw_trials.median().median() for v in sel.values()]
print("\nmedian trials per session by subject, under other selections (expected 110 +- 11; Switch trials 54 +- 5.4):")
print(tab.round(2).to_string())

**Result: partially reproduced.** The median of the five subjects' median session accuracies is 73.0%, within 72.6 +- 7.26; the median of subject medians of trials per session is 152.0, outside 110 +- 11 (pooled sessions give 76.4% and 150.5). The repository's own perf_table.csv gives the same 152 under the same rule, and no trial threshold from 40 to 60 brings it near 110 (135 to 152). No other selection in the table of selections gives it either: the median of subject medians ranges from 133.5 (all sessions) to 190.0 (at least 100 trials), the mean of subject medians from 125.2 to 194.5, and the Switch trials from 64.0 to 95.0, against 54 +- 5.4.

## fig1-switch-performance

> Performance on Switch trials (cue differs from the previous trial) was similar to non-Switch trials,
> with a median Switch-trial accuracy of 76.9% and a median of 54 Switch trials per session.

In [ ]:
recomputed_sw = all((r["trials"].sw.values[1:] == (r["trials"].cue.values[1:] != r["trials"].cue.values[:-1])).all()
                    for r in meta if "trials" in r)
print("sw column equals 'cue differs from the previous trial' in every session:", recomputed_sw)
sub_nsw = incl.groupby("subject").pct_nsw_correct.median()
sw_acc, n_sw = subj_med.pct_sw_correct.median() * 100, subj_med.n_sw_trials.median()
nsw_acc = sub_nsw.median() * 100
print(f"median of subject medians: Switch accuracy {sw_acc:.1f}% (expected 76.9 +- 7.69), "
      f"Switch trials {n_sw:.1f} (expected 54 +- 5.4), non-Switch accuracy {nsw_acc:.1f}%, "
      f"difference {sw_acc - nsw_acc:.1f} points (criterion: within 10)")
print(f"variant, pooled: Switch accuracy {incl.pct_sw_correct.median() * 100:.1f}%, Switch trials "
      f"{incl.n_sw_trials.median():.1f}, non-Switch {incl.pct_nsw_correct.median() * 100:.1f}%")
w = stats.wilcoxon(incl.pct_sw_correct, incl.pct_nsw_correct)
print(f"per session Switch minus non-Switch accuracy: median {(incl.pct_sw_correct - incl.pct_nsw_correct).median() * 100:.1f} "
      f"points, Wilcoxon p = {w.pvalue:.3g} (not the paper's test)")
flag = incl.groupby("subject")[["pct_sw_correct", "pct_sw_correct_flagged", "pct_nsw_correct"]].median()
print("\nper subject, medians over sessions: Switch accuracy as the code computes it (all Switch trials), over the "
      "Switch trials with a correct flag (variant), and non-Switch accuracy")
print((flag * 100).round(1).to_string())
fl = flag.pct_sw_correct_flagged.median() * 100
print(f"variant, Switch trials with a correct flag only: median of subject medians {fl:.1f}%, difference from "
      f"non-Switch {fl - nsw_acc:.1f} points (criterion: within 10); pooled {incl.pct_sw_correct_flagged.median() * 100:.1f}% "
      f"against {incl.pct_nsw_correct.median() * 100:.1f}%, difference "
      f"{(incl.pct_sw_correct_flagged.median() - incl.pct_nsw_correct.median()) * 100:.2f} points")
print("Switch trials without a correct flag per included session: median",
      float(np.median([meta_by_session[s]["trials"].query("sw == 1").correct.isna().sum() for s in incl.index])),
      "; total", int(sum(meta_by_session[s]["trials"].query("sw == 1").correct.isna().sum() for s in incl.index)))

**Result: partially reproduced.** The median Switch-trial accuracy is 73.8% (expected 76.9 +- 7.69) and lies 2.4 points above the non-Switch accuracy of 71.4%, so the similarity holds under the 10-point threshold the claims file fixes; the median number of Switch trials per session is 69.0, outside 54 +- 5.4. The `sw` column matches the cue-change definition in every session. Switch accuracy is computed as the code computes it (all Switch trials in the denominator, change-of-mind trials included; 69 such trials over the included sessions, a median of 0 per session). Over the Switch trials with a correct flag only, the median of subject medians is 75.2%, 3.8 points above non-Switch, so the similarity still holds; pooled over sessions that variant gives 81.6% against 71.6%, a difference of 10.00 points, at the threshold.

## Remapping scores and segment rate comparisons

Computed once for the claims that use them. Sessions: the included sessions (at least one unit, at least
50 trials) without the session the code marks invalid. The single-unit and population scores are those of
the opening comment of the claims file (code: `all_zone_remapping_analyses`, `pop_zone_remapping_analyses`),
with 100 balanced resamples. The cue score's generator is seeded from fig2d-cue-remapping and the reward
score's from fig4d-reward-remapping; the claims that use the scores use those seeds. Each score is also
computed under the seed plus 1000 to 4000, under the code's seed 0, and, for the reward score, with the null
balanced for correct and incorrect trials (the Figure 4d paragraph's description). The control scores (Pearson
correlation; 500 ms after reward blanked; speeds outside 2 to 100 cm/s blanked) are seeded from the control
claims, fig2-remap-behaviour-controls and fig4-reward-remap-behaviour-controls, under that seed only.

In [ ]:
SEED_CUE, SEED_RW = seed_of("fig2d-cue-remapping"), seed_of("fig4d-reward-remapping")
SEED_CTL = {"cue": seed_of("fig2-remap-behaviour-controls"), "rw": seed_of("fig4-reward-remap-behaviour-controls")}
OFFS = [0, 1000, 2000, 3000, 4000]
use = [s for s in incl.index if s not in tma.INVALID_SESSIONS and s in TM]
print("sessions used for the neural analyses:", len(use), "; left out as invalid:",
      [s for s in incl.index if s in tma.INVALID_SESSIONS])
t0 = time.time()
unit_rows, pop_rows, uz_rows = [], [], []
for s in use:
    d = TM[s]
    ses = d["ses"]
    u = d["units"].reset_index(drop=True)
    base = dict(session=s, subject=d["subject"], task=s.split("_")[1])
    sc = {}
    for comp, seed in [("cue", SEED_CUE), ("rw", SEED_RW)]:
        for off in OFFS:
            sc[f"{comp}_s{off}"] = tma.remap_scores(ses, comp, seed + off)
        sc[f"{comp}_seed0"] = tma.remap_scores(ses, comp, 0)
        sc[f"{comp}_pearson"] = tma.remap_scores(ses, comp, SEED_CTL[comp], method="pearson")
        sc[f"{comp}_rwblank"] = tma.remap_scores(d["ses_rw"], comp, SEED_CTL[comp])
        sc[f"{comp}_spblank"] = tma.remap_scores(d["ses_sp"], comp, SEED_CTL[comp])
    sc["rw_null_co"] = tma.remap_scores(ses, "rw", SEED_RW, bal=tma.BAL_RW_NULL_CO)
    df = pd.DataFrame(sc)
    df["unit"] = np.arange(len(u))
    df["unit_type"] = u.unit_type.values
    for k, v in base.items():
        df[k] = v
    unit_rows.append(df)
    for comp, seed in [("cue", SEED_CUE), ("rw", SEED_RW)]:
        for grp, idx in [("all", np.arange(len(u))), ("cell", np.where(u.unit_type == "cell")[0])]:
            if len(idx) == 0:
                continue
            row = dict(base, comp=comp, pop_type=grp, n_units=len(idx))
            for off in OFFS:
                row[f"s{off}"] = tma.pop_remap_score(ses, comp, seed + off, units=idx)
            pop_rows.append(row)
    for comp in ["cue", "rw"]:
        z = tma.segment_uz(ses, comp)
        for j, segn in enumerate(["left", "stem", "right"]):
            uz_rows.append(pd.DataFrame(dict(base, comp=comp, unit=np.arange(len(u)), unit_type=u.unit_type.values,
                                             segment=segn, uz=z[:, j])))
SC = pd.concat(unit_rows, ignore_index=True)
POP = pd.DataFrame(pop_rows)
UZ = pd.concat(uz_rows, ignore_index=True)
UZ["segment"] = pd.Categorical(UZ.segment, ["left", "stem", "right"])
for c in ["pct_correct"]:
    SC[c] = SC.session.map(BEH[c])
    POP[c] = POP.session.map(BEH[c])
print(f"computed in {time.time() - t0:.0f} s; units {len(SC)} (cell {(SC.unit_type == 'cell').sum()}, "
      f"mua {(SC.unit_type == 'mua').sum()}) in {SC.session.nunique()} sessions")
print("units with a cue score:", int(SC.cue_s0.notna().sum()), "; with a reward score:", int(SC.rw_s0.notna().sum()))
print("sessions without a cue score (an empty or small balanced set):", SC[SC.cue_s0.isna()].session.unique().tolist())
print("sessions without a reward score:", SC[SC.rw_s0.isna()].session.unique().tolist())
print("score ranges:", SC[["cue_s0", "rw_s0"]].describe().loc[["min", "50%", "max"]].round(2).to_dict())

In [ ]:
def remap_lmem(df, col, unit_set):
    """paper_stats.py, _remap_scores: score ~ 1 (+ unit_type when MUA are in), groups subject,
    variance components task version and session; REML."""
    d = df[["subject", "session", "task", "unit_type", col]].dropna().rename(columns={col: "score"})
    if unit_set == "cell":
        d = d[d.unit_type == "cell"]
    vc = {"task": "1+C(task)", "session": "0+C(session)"}
    f = "score ~ 1 + unit_type" if unit_set == "all" else "score ~ 1"
    m = tma.mixed(f, d, vc)
    ci = m.conf_int().loc["Intercept"].values
    return dict(n=len(d), mean=m.params["Intercept"], ci_low=ci[0], ci_high=ci[1], p=m.pvalues["Intercept"],
                raw_mean=d.score.mean())


def beh_lmem(df, col, unit_set, session_mean=False, rescale=True):
    """paper_stats.py, _unit_remap_beh: logit(performance) ~ 1 + remap, groups subject, variance
    component task version (and unit type with MUA, single units); LRT against ~ 1 by ML."""
    d = df[["subject", "session", "task", "unit_type", col, "pct_correct"]].dropna()
    d = d.rename(columns={col: "remap", "pct_correct": "behav"})
    if unit_set == "cell":
        d = d[d.unit_type == "cell"]
    if session_mean:
        d = d.groupby(["subject", "task", "session"])[["remap", "behav"]].mean().reset_index()
    tau = stats.kendalltau(d.behav, d.remap)
    if rescale:
        d["behav"] = tma.logit(d["behav"])
    vc = {"task": "1+C(task)"}
    if unit_set == "all" and not session_mean:
        vc["unit"] = "0+C(unit_type)"
    m = tma.mixed("behav ~ 1 + remap", d, vc)
    chi, dof, p, refit = tma.lrt_models("behav ~ 1 + remap", "behav ~ 1", d, vc)
    ci = m.conf_int().loc["remap"].values
    return dict(n=len(d), tau=tau[0], tau_p=tau[1], slope=m.params["remap"], ci_low=ci[0], ci_high=ci[1],
                lrt=chi, df=dof, p=p, refit=refit)


def show(rows):
    print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))

## fig2c-cue-error-coding

> Across units, the Mann-Whitney z of Right-Cue against Left-Cue segment firing rates on outbound runs
> follows a left > stem > right pattern, so firing is higher on the branch the cue did not indicate.

In [ ]:
def segment_lmem(comp, unit_set):
    """paper_stats.py, _segment_rates: score ~ 0 + segment, groups subject, variance components task
    version and session (and unit type with MUA); LRT against score ~ 1 by ML; Wald test of the
    ordering contrasts on the REML fit."""
    d = UZ[UZ.comp == comp].dropna(subset=["uz"]).rename(columns={"uz": "score"})
    if unit_set == "cell":
        d = d[d.unit_type == "cell"]
    vc = {"task": "1+C(task)", "session": "0+C(session)"}
    if unit_set == "all":
        vc["unit"] = "0+C(unit_type)"
    m = tma.mixed("score ~ 0 + segment", d, vc)
    chi, dof, p, refit = tma.lrt_models("score ~ 0 + segment", "score ~ 1", d, vc)
    names = list(m.fe_params.index)
    il, is_, ir = names.index("segment[left]"), names.index("segment[stem]"), names.index("segment[right]")
    Rm = np.zeros((2, len(names)))
    Rm[0, [il, is_]] = 1, -1
    Rm[1, [is_, ir]] = 1, -1
    w = m.wald_test(np.hstack([Rm, np.zeros((2, len(m.params) - len(names)))]), scalar=True)
    t = m.t_test(Rm)
    return dict(unit_set=unit_set, n_rows=len(d), n_units=d.groupby(["session", "unit"]).ngroups,
                left=m.fe_params.iloc[il], stem=m.fe_params.iloc[is_], right=m.fe_params.iloc[ir],
                lrt=chi, df=dof, p_lrt=p, refit=refit, p_lrt_df3=stats.chi2.sf(chi, 3), wald=float(np.squeeze(w.statistic)),
                p_wald=float(w.pvalue), p_left_gt_stem_1s=t.pvalue[0] / 2 if t.effect[0] > 0 else 1 - t.pvalue[0] / 2,
                p_stem_gt_right_1s=t.pvalue[1] / 2 if t.effect[1] > 0 else 1 - t.pvalue[1] / 2)

In [ ]:
print("U_Z (CR against CL, outbound) by segment, mean and median over units:")
print(UZ[UZ.comp == "cue"].groupby(["unit_type", "segment"]).uz.agg(["count", "mean", "median"]).round(3).to_string())
print("units with no U_Z in a segment (a condition with no trial of 5 samples there):",
      int(UZ[UZ.comp == "cue"].uz.isna().sum()))
show([segment_lmem("cue", "cell"), segment_lmem("cue", "all")])
print("\nby subject, mean U_Z (all units):")
print(UZ[UZ.comp == "cue"].groupby(["subject", "segment"]).uz.mean().unstack().round(3).to_string())

**Result: reproduced.** For isolated units the segment effects are left 0.600, stem 0.285, right -0.226 (LRT 89.3, df 2, p = 4.0e-20; Wald test of the two ordering contrasts 89.0, p = 4.8e-20; one-sided left > stem p = 1.8e-4, stem > right p = 3.6e-9). With MUA: 0.556, 0.325, -0.228, LRT 276.8, p = 7.7e-61. By subject (all units), left > stem > right holds in Cl and Li; in Al, Ca and Ne the stem mean is above the left, and the right is lowest in all five.

## fig2d-cue-remapping

> Across units the cue remapping score is shifted below zero (mean -0.67), so the cue induced significant
> remapping of spatial firing compared with an even and odd trial null.

In [ ]:
rows = []
for us in ["cell", "all"]:
    for off in OFFS:
        rows.append(dict(unit_set=us, seed=f"claim+{off}", **remap_lmem(SC, f"cue_s{off}", us)))
    rows.append(dict(unit_set=us, seed="code seed 0", **remap_lmem(SC, "cue_seed0", us)))
show(rows)
print("expected mean -0.67 +- 0.44, CI excluding zero")
print("\nby subject, median score (claim seed):")
print(SC.groupby(["subject", "unit_type"]).cue_s0.agg(["count", "median"]).round(3).unstack().to_string())

**Result: reproduced.** The LMEM mean cue remapping score is -0.540 for isolated units (95% CI -0.793 to -0.288, n = 507) and -0.551 with MUA (CI -0.783 to -0.318, n = 1732), both within -0.67 +- 0.44 with CIs below zero. Over the five seeds the mean ranges from -0.596 to -0.540 (isolated) and -0.576 to -0.551 (with MUA); the code's seed 0 gives -0.559 and -0.573.

## fig2e-unit-remap-behaviour

> The cue remapping score of single units correlates negatively with the session's task performance.

In [ ]:
rows = []
for us in ["cell", "all"]:
    for off in OFFS:
        rows.append(dict(unit_set=us, seed=f"claim+{off}", **beh_lmem(SC, f"cue_s{off}", us)))
    rows.append(dict(unit_set=us, seed="raw fraction, claim", **beh_lmem(SC, "cue_s0", us, rescale=False)))
show(rows)
print("expected tau -0.27 +- 0.027, slope -0.12 +- 0.03 (logit scale), LRT p < 0.05")

**Result: reproduced.** For isolated units Kendall tau is -0.266 (expected -0.27 +- 0.027) and the LMEM slope on logit performance is -0.145 (CI -0.188 to -0.102; expected -0.12 +- 0.03), LRT 42.6, p = 6.6e-11. Over the five seeds tau ranges from -0.269 to -0.261 and the slope from -0.147 to -0.142, all within tolerance. With MUA, tau is -0.231 (outside tolerance) and the slope -0.124 (inside), LRT p = 3.9e-25. On the raw fraction the slope is -0.026.

## fig2f-session-remap-behaviour

> The mean cue remapping score of co-recorded units in a session correlates more strongly with task
> performance than single-unit scores do.

In [ ]:
rows = []
for us in ["cell", "all"]:
    for off in OFFS:
        r1 = beh_lmem(SC, f"cue_s{off}", us, session_mean=True)
        r0 = beh_lmem(SC, f"cue_s{off}", us)
        rows.append(dict(unit_set=us, seed=f"claim+{off}", **r1, unit_tau=r0["tau"]))
show(rows)
print("expected tau -0.41 +- 0.041, slope -0.22 +- 0.11, LRT p < 0.05, |tau| above the unit-level |tau|")

**Result: partially reproduced.** With session means of isolated units, tau is -0.422 (within -0.41 +- 0.041; unit level -0.266, so stronger) and the LRT gives 17.3, p = 3.1e-5, but the slope is -0.385 (CI -0.561 to -0.209), outside -0.22 +- 0.11. With MUA tau is -0.376 (within) and the slope -0.384 (outside). Over five seeds the isolated-unit slope ranges from -0.385 to -0.361, all outside.

## fig2g-population-remap-behaviour

> Population-level remapping, from the correlation of the session's population spatial rate maps, shows
> the same negative relation to task performance.

In [ ]:
def pop_beh_lmem(comp, pop_type, col, rescale=True):
    """paper_stats.py, _pop_remap_beh: logit(performance) ~ 1 + remap, groups subject, variance
    component task version; LRT by ML."""
    d = POP[(POP.comp == comp) & (POP.pop_type == pop_type)][["subject", "session", "task", col, "pct_correct"]]
    d = d.dropna().rename(columns={col: "remap", "pct_correct": "behav"})
    tau = stats.kendalltau(d.behav, d.remap)
    if rescale:
        d["behav"] = tma.logit(d.behav)
    vc = {"task": "1+C(task)"}
    m = tma.mixed("behav ~ 1 + remap", d, vc)
    chi, dof, p, refit = tma.lrt_models("behav ~ 1 + remap", "behav ~ 1", d, vc)
    ci = m.conf_int().loc["remap"].values
    return dict(n_sessions=len(d), mean_score=d.remap.mean(), tau=tau[0], tau_p=tau[1], slope=m.params["remap"],
                ci_low=ci[0], ci_high=ci[1], lrt=chi, df=dof, p=p, refit=refit)

In [ ]:
rows = []
for pt in ["all", "cell"]:
    for off in OFFS:
        rows.append(dict(pop_type=pt, seed=f"claim+{off}", **pop_beh_lmem("cue", pt, f"s{off}")))
show(rows)
print("criterion: LRT p < 0.05 and negative slope (population of all units)")

**Result: reproduced.** The population cue remapping score of all units relates negatively to logit performance: slope -0.104 (CI -0.144 to -0.065), LRT 25.0, p = 5.7e-7, tau -0.376, over 122 sessions; isolated-unit populations give slope -0.077, p = 8.3e-6. All five seeds give p < 2e-6 with a negative slope.

## fig2h-subject-remap-behaviour

> The negative correlation between task performance and session mean cue remapping holds in all subjects
> but one, which had few units.

In [ ]:
def subject_corrs(col, claim, n_boot=500):
    """plot_functions.py, plot_subject_remap_beh_slope: per subject, Kendall correlation of session mean
    remapping and logit performance, bootstrapped over the subject's sessions (500 resamples)."""
    d = SC[["subject", "task", "session", col, "pct_correct"]].dropna()
    d = d.groupby(["subject", "task", "session"])[[col, "pct_correct"]].mean().reset_index()
    d["y"] = tma.logit(d.pct_correct)
    units = SC.dropna(subset=[col]).groupby("subject").size()
    out = []
    for i, off in enumerate(OFFS):
        rng = np.random.RandomState(seed_of(claim) + off)
        for s in ["Li", "Ne", "Cl", "Al", "Ca"]:
            sub = d[d.subject == s]
            x, y = sub[col].values, sub.y.values
            rb = np.array([stats.kendalltau(x[ix], y[ix])[0] for ix in
                           (rng.choice(len(x), len(x)) for _ in range(n_boot))])
            out.append(dict(seed=f"claim+{off}", subject=s, sessions=len(sub), units=int(units.get(s, 0)),
                            tau=stats.kendalltau(x, y)[0], boot_mean=np.nanmean(rb), boot_sd=np.nanstd(rb)))
    return pd.DataFrame(out)

In [ ]:
H = subject_corrs("cue_s0", "fig2h-subject-remap-behaviour")
print(H[H.seed == "claim+0"].round(3).to_string())
print("\nnegative point correlations per seed of the bootstrap (the point value does not depend on it):")
print(H.groupby("seed").apply(lambda g: pd.Series(dict(neg_tau=int((g.tau < 0).sum()),
                                                           neg_boot_mean=int((g.boot_mean < 0).sum())))).to_string())
print("subject with fewest units:", H.loc[H.units.idxmin(), "subject"])

**Result: reproduced.** All five subjects have a negative Kendall correlation between session mean cue remapping and logit performance (Li -0.418, Ne -0.380, Cl -0.483, Al -0.289, Ca -0.143), under every seed of the bootstrap; Ca, the subject with fewest units (29), has the weakest, with bootstrap SD 0.216. The criterion allows one exception and is met; the paper's exception does not appear.

## fig2-remap-behaviour-controls

> The relation between cue remapping and behaviour holds when the remapping score uses Pearson in place of
> Kendall correlation, when 500 ms after reward delivery are removed, and when periods of speed below
> 2 cm/s are removed, for isolated units and with MUA.

In [ ]:
def control_rows(comp):
    rows = []
    for ctl, col in [("pearson", f"{comp}_pearson"), ("reward blank", f"{comp}_rwblank"),
                     ("speed blank", f"{comp}_spblank")]:
        for us in ["cell", "all"]:
            for lvl, sm in [("unit", False), ("session mean", True)]:
                r = beh_lmem(SC, col, us, session_mean=sm)
                rows.append(dict(control=ctl, unit_set=us, level=lvl, **r))
    out = pd.DataFrame(rows)
    out["holds"] = (out.p < 0.05) & (out.slope < 0)
    return out

In [ ]:
CT = control_rows("cue")
print(CT.to_string(float_format=lambda x: f"{x:.4g}"))
print(f"\ncontrol tests with LRT p < 0.05 and negative slope: {int(CT.holds.sum())} of {len(CT)}")
print("blanked samples, median fraction per session: reward",
      np.median([TM[s]["ses_rw"].n_blank / TM[s]["ses_rw"].n_samps for s in use]).round(3),
      "; speed", np.median([TM[s]["ses_sp"].n_blank / TM[s]["ses_sp"].n_samps for s in use]).round(3))
BLK = pd.DataFrame({s: dict(subject=TM[s]["subject"], default=TM[s]["ses"].n_blank / TM[s]["ses"].n_samps,
                            reward=TM[s]["ses_rw"].n_blank / TM[s]["ses_rw"].n_samps,
                            speed=TM[s]["ses_sp"].n_blank / TM[s]["ses_sp"].n_samps) for s in use}).T
BLK[["default", "reward", "speed"]] = BLK[["default", "reward", "speed"]].astype(float)
print("\nblanked fraction of samples per session (default blanking, with the reward control, with the speed control), "
      "quantiles over", len(BLK), "sessions:")
print(BLK[["default", "reward", "speed"]].quantile([0, 0.1, 0.25, 0.5, 0.75, 0.9, 1]).round(3).to_string())
print("median per subject:")
print(BLK.groupby("subject")[["default", "reward", "speed"]].median().round(3).to_string())

**Result: reproduced.** All 12 control tests (Pearson score, reward blanking, speed blanking; isolated units and with MUA; unit and session-mean levels) give an LRT p below 2e-4 with a negative slope (slopes -0.107 to -0.410). The control scores were computed under this claim's seed only. Speed blanking removes 33% to 68% of the samples of a session (median 45%; subject medians 39% to 54%), reward blanking 4% to 16% (median 9%), and the default blanking a median 0.5%.

## fig4c-reward-error-coding

> On inbound runs, unrewarded trials tend to have higher firing rates than rewarded trials in all maze
> segments (left 386/565, stem 340/565, right 412/565 units), with a significant effect of segment.

In [ ]:
rw = UZ[UZ.comp == "rw"].copy()
rw["le0"] = rw.uz <= 0
for ut in ["cell", "mua"]:
    t = rw[rw.unit_type == ut]
    # paper_stats.py, _segment_rates: crosstab of (uz <= 0) by segment within one unit type; a NaN counts as not <= 0
    ct = t.groupby("segment").agg(n_le0=("le0", "sum"), n_rows=("le0", "size"), n_valid=("uz", "count"))
    ct["frac_of_rows"] = ct.n_le0 / ct.n_rows
    print(ut)
    print(ct.to_string())
print("expected (isolated units): left 386, stem 340, right 412 of 565 (tolerance 10%), each above half")
show([segment_lmem("rw", "cell"), segment_lmem("rw", "all")])
print("\nby subject, fraction of isolated units with U_Z <= 0:")
print(rw[rw.unit_type == "cell"].groupby(["subject", "segment"]).le0.mean().unstack().round(3).to_string())

**Result: reproduced.** Among 510 isolated units, U_Z <= 0 (unrewarded above rewarded) in 357 left, 326 stem and 374 right, each within 10% of 386, 340 and 412 and above half (70%, 64%, 73%); the paper counts 565 isolated units. The segment effect has LRT 45.3, df 2, p = 1.4e-10 (with MUA 113.1, p = 2.8e-25). MUA alone give 828, 762 and 862 of 1234.

## fig4d-reward-remapping

> Across units the reward remapping score (rewarded against unrewarded inbound maps, against an even and odd
> null) is shifted below zero (mean -1.03).

In [ ]:
rows = []
for us in ["cell", "all"]:
    for off in OFFS:
        rows.append(dict(unit_set=us, null="even/odd balanced for cue", seed=f"claim+{off}",
                         **remap_lmem(SC, f"rw_s{off}", us)))
    rows.append(dict(unit_set=us, null="even/odd balanced for cue", seed="code seed 0", **remap_lmem(SC, "rw_seed0", us)))
    rows.append(dict(unit_set=us, null="even/odd balanced for correct", seed="claim+0", **remap_lmem(SC, "rw_null_co", us)))
show(rows)
print("expected mean -1.03 +- 0.57, CI excluding zero")

**Result: reproduced.** The LMEM mean reward remapping score is -1.022 for isolated units (CI -1.544 to -0.500) and -1.038 with MUA (CI -1.419 to -0.656), within -1.03 +- 0.57; five seeds give -1.034 to -1.007 (isolated). With the null balanced for correct and incorrect trials, as the Figure 4d paragraph describes, the means are -0.606 (CI -1.039 to -0.173) and -0.634, also within tolerance with CIs below zero.

## fig4ef-reward-remap-behaviour

> Reward remapping correlates negatively with session task performance at the level of single units and
> of session means across co-recorded units.

In [ ]:
rows = []
for us in ["cell", "all"]:
    for off in OFFS:
        a = beh_lmem(SC, f"rw_s{off}", us)
        b = beh_lmem(SC, f"rw_s{off}", us, session_mean=True)
        rows.append(dict(unit_set=us, seed=f"claim+{off}", unit_tau=a["tau"], unit_slope=a["slope"],
                         unit_ci=(round(a["ci_low"], 3), round(a["ci_high"], 3)), unit_lrt=a["lrt"], unit_p=a["p"],
                         ses_n=b["n"], ses_tau=b["tau"], ses_slope=b["slope"],
                         ses_ci=(round(b["ci_low"], 3), round(b["ci_high"], 3)), ses_lrt=b["lrt"], ses_p=b["p"]))
show(rows)
print("expected unit tau -0.28 +- 0.028, unit slope -0.14 +- 0.035, session tau -0.43 +- 0.043, "
      "session slope -0.28 +- 0.095")

**Result: partially reproduced.** Both LRTs have p < 0.05 with negative slopes under both unit sets, but not all four values are within tolerance under either. Isolated units: unit tau -0.242 (outside -0.28 +- 0.028), unit slope -0.152 (inside), session tau -0.359 (outside -0.43 +- 0.043), session slope -0.318 (inside); unit LRT 83.8, p = 5.5e-20; session LRT 34.9, p = 3.5e-9. With MUA: -0.239 (outside), -0.135 (inside), -0.411 (inside), -0.412 (outside -0.28 +- 0.095).

## fig4gh-reward-remap-population-subject

> Reward remapping also relates negatively to session performance at the population level and within
> individual subjects.

In [ ]:
rows = []
for pt in ["all", "cell"]:
    for off in OFFS:
        rows.append(dict(pop_type=pt, seed=f"claim+{off}", **pop_beh_lmem("rw", pt, f"s{off}")))
show(rows)
H4 = subject_corrs("rw_s0", "fig4gh-reward-remap-population-subject")
print(H4[H4.seed == "claim+0"].round(3).to_string())
print(H4.groupby("seed").apply(lambda g: pd.Series(dict(neg_tau=int((g.tau < 0).sum()),
                                                            neg_boot_mean=int((g.boot_mean < 0).sum())))).to_string())

**Result: reproduced.** The population reward remapping score of all units relates negatively to performance (slope -0.107, LRT 41.8, p = 1.0e-10, 122 sessions; five seeds p below 3e-10), and all five subjects have a negative correlation (Li -0.579, Ne -0.400, Cl -0.418, Al -0.378, Ca -0.448).

## fig4-reward-remap-behaviour-controls

> The relation between reward remapping and behaviour holds when the remapping score uses Pearson in place of
> Kendall correlation, when 500 ms after reward delivery are removed, and when periods of speed below 2 cm/s
> are removed, for isolated units and with MUA.

In [ ]:
CT4 = control_rows("rw")
print(CT4.to_string(float_format=lambda x: f"{x:.4g}"))
print(f"\ncontrol tests with LRT p < 0.05 and negative slope: {int(CT4.holds.sum())} of {len(CT4)}")

**Result: reproduced.** All 12 control tests give an LRT p below 4e-9 with a negative slope (slopes -0.117 to -0.420). The control scores were computed under this claim's seed only.

## fig4s5-cue-reward-remap-correlated

> Cue and reward remapping scores are correlated across units, across session means and at the population
> level.

In [ ]:
rows = []
for us in ["cell", "all"]:
    for off in OFFS:
        d = SC if us == "all" else SC[SC.unit_type == "cell"]
        d = d[["session", f"cue_s{off}", f"rw_s{off}"]].dropna()
        a = stats.kendalltau(d[f"cue_s{off}"], d[f"rw_s{off}"])
        m = d.groupby("session").mean()
        b = stats.kendalltau(m[f"cue_s{off}"], m[f"rw_s{off}"])
        pp = POP[POP.pop_type == us].pivot_table(index="session", columns="comp", values=f"s{off}").dropna()
        c = stats.kendalltau(pp.cue, pp.rw)
        rows.append(dict(unit_set=us, seed=f"claim+{off}", unit_tau=a[0], unit_p=a[1], unit_n=len(d),
                         session_tau=b[0], session_p=b[1], session_n=len(m), pop_tau=c[0], pop_p=c[1], pop_n=len(pp)))
show(rows)

**Result: reproduced.** Cue and reward remapping scores are positively correlated for isolated units at the unit (tau 0.211, p = 1.1e-12, n = 507), session-mean (0.374, p = 1.8e-7, 90 sessions) and population (0.447, p = 4.5e-10) levels, and with MUA (0.198, 0.414, 0.581, all p < 2e-11), under all five seeds.

## fig4s6-incorrect-coding-overlap

> Units coding for incorrect trials overlap more between cue (outbound) and reward (inbound) than units
> coding for correct trials, while remapping scores do not differ among these groupings.

In [ ]:
W = UZ.pivot_table(index=["session", "unit", "unit_type"], columns=["comp", "segment"], values="uz", observed=False)
cue_u = W["cue"][["left", "stem", "right"]].dropna()
rw_u = W["rw"][["left", "stem", "right"]].dropna()
# experiment_info.py: fr_uz_tau_cue_correct (Kendall tau of left, stem, right U_Z with -1, 0, 1) and
# fr_uz_tau_rw_correct ((number of segments with U_Z > 0 - 1.5) / 1.5)
tau_cue = pd.Series(tma.kendall_tau_b(cue_u.values, np.tile([-1.0, 0.0, 1.0], (len(cue_u), 1))), index=cue_u.index)
tau_rw = ((rw_u > 0).sum(axis=1) - 1.5) / 1.5
G = pd.DataFrame(dict(cue=tau_cue, rw=tau_rw)).dropna()
G = G.join(SC.set_index(["session", "unit", "unit_type"])[["cue_s0", "rw_s0"]]).dropna(subset=["cue_s0"])
print("cue coding score values:", G.cue.round(3).value_counts().sort_index().to_dict())
print("reward coding score values:", G.rw.round(3).value_counts().sort_index().to_dict())
rows = []
for us in ["all", "cell"]:
    g = G if us == "all" else G[G.index.get_level_values("unit_type") == "cell"]
    for crit, (co, inco) in {"strict": (lambda v: np.isclose(v, 1), lambda v: np.isclose(v, -1)),
                             "liberal": (lambda v: v > 0, lambda v: v < 0)}.items():
        sets = {}
        for lab, f in [("correct", co), ("incorrect", inco)]:
            a, b = f(g.cue), f(g.rw)
            sets[lab] = dict(both=int((a & b).sum()), either_only=int((a ^ b).sum()), cue=int(a.sum()), rw=int(b.sum()))
        table = [[sets["incorrect"]["both"], sets["incorrect"]["either_only"]],
                 [sets["correct"]["both"], sets["correct"]["either_only"]]]
        o2, p2 = stats.fisher_exact(table)
        o1, p1 = stats.fisher_exact(table, alternative="greater")
        rows.append(dict(unit_set=us, criterion=crit, n=len(g), inco_both=table[0][0], inco_one=table[0][1],
                         co_both=table[1][0], co_one=table[1][1], odds=o2, p_two_sided=p2, p_greater=p1))
show(rows)
print("expected Fisher p 0.003 (strict) and 0.007 (liberal), incorrect overlap greater")

G = G.join(SC.set_index(["session", "unit", "unit_type"])[[f"{c}_s{o}" for c in ["cue", "rw"] for o in OFFS[1:]]])
ks = []
for us in ["all", "cell"]:
    g = G if us == "all" else G[G.index.get_level_values("unit_type") == "cell"]
    for code, score in [("cue", "cue_s0"), ("rw", "rw_s0")]:
        grp = g[code].round(3)
        levels = sorted(grp.unique())
        for i in range(len(levels)):
            for j in range(i + 1, len(levels)):
                x, y = g.loc[grp == levels[i], score], g.loc[grp == levels[j], score]
                if len(x) and len(y):
                    r = stats.ks_2samp(x, y)
                    ks.append(dict(unit_set=us, coding=code, g1=levels[i], g2=levels[j], n1=len(x), n2=len(y),
                                   median1=x.median(), median2=y.median(), D=r.statistic, p=r.pvalue))
KS = pd.DataFrame(ks)
print(KS.to_string(float_format=lambda x: f"{x:.4g}"))
for us in ["all", "cell"]:
    k = KS[KS.unit_set == us]
    print(f"{us}: KS tests with p < 0.05: {int((k.p < 0.05).sum())} of {len(k)}; smallest p {k.p.min():.3g}")
print("isolated units, the same 12 KS tests on the scores of the other four seeds:")
for off in OFFS[1:]:
    g = G[G.index.get_level_values("unit_type") == "cell"]
    ps = []
    for code in ["cue", "rw"]:
        grp = g[code].round(3)
        lv = sorted(grp.unique())
        ps += [stats.ks_2samp(g.loc[grp == lv[i], f"{code}_s{off}"].dropna(), g.loc[grp == lv[j], f"{code}_s{off}"].dropna()).pvalue
               for i in range(len(lv)) for j in range(i + 1, len(lv))]
    print(f"  seed+{off}: p < 0.05 in {sum(p < 0.05 for p in ps)} of {len(ps)}; smallest p {min(ps):.3g}")
for us in ["all", "cell"]:
    g = G if us == "all" else G[G.index.get_level_values("unit_type") == "cell"]
    inco_one = (g.cue < 0) ^ (g.rw < 0)
    co_one = (g.cue > 0) ^ (g.rw > 0)
    print(f"{us}, liberal: units incorrect in one coding only {int(inco_one.sum())}, correct in one only "
          f"{int(co_one.sum())}, the same units: {bool((inco_one == co_one).all())}; units with a zero coding score: "
          f"{int(((g.cue == 0) | (g.rw == 0)).sum())}; so the table compares {int(((g.cue < 0) & (g.rw < 0)).sum())} "
          f"units incorrect in both with {int(((g.cue > 0) & (g.rw > 0)).sum())} correct in both, against the same "
          f"{int(inco_one.sum())} mixed units")

**Result: reproduced.** On isolated units, the unit set of the code's figure function (`plot_cue_rw_correct_rate_coding_overlap`, unit_type 'cell'), units coding incorrect overlap more between cue and reward than units coding correct: strict 69 of 263 against 6 of 65 (Fisher p = 0.0028), liberal 263 of 449 against 58 of 244 (p = 6.1e-19). None of the 12 two-sample KS tests of remapping scores between the four coding groups has p < 0.05 (smallest 0.073) under the claim's seed; under the four other seeds 2, 0, 1 and 0 of the 12 have p < 0.05 (smallest 0.030), so the absence part depends on the seed. With MUA included, the Fisher tests hold (p = 4.7e-13 and 3.1e-58) but 5 of 12 KS tests have p < 0.05 (smallest 0.0070). In the liberal table every unit has a nonzero coding score, so the 186 units incorrect in one coding only are the same 186 units that are correct in one only (all units: 609 and 609); the liberal test thus compares 263 units incorrect in both with 58 correct in both, against the same mixed units, and adds little to the strict test.

## fig4s7-error-signal-persists

> Higher activity on incorrect than correct outbound runs predicts higher activity on the following incorrect
> inbound runs, and the reverse inference also holds.

The claims file gives the measure (z-scored mean rates of correct and incorrect trials per unit and branch),
the session rule and the model terms. The rest is taken from the paper's code (author, step 1.6):
`tree_maze_functions.py, mean_segment_rates_analysis` (per unit, condition, run and big segment, the mean
over trials of the trial segment rate divided by its standard deviation, the "z"), and
`experiment_info.py, get_conditional_correct_bigseg_rates` (units with more than 5 incorrect outbound trials
with a stem rate; branches left and right; `zo_p` marks a unit whose correct minus incorrect z is above 0 on
an outbound branch, `zi_p` the same inbound). `paper_stats.py, _conditiona_co_inco_rates`: for inbound runs,
z ~ ms + zo_p + cond against z ~ ms + zo_p (main effect of incorrect), and on the correct minus incorrect
differences, z ~ ms + zo_p against z ~ ms (the conditioning effect, the legend's interaction); the reverse
for outbound runs; groups subject, variance components task version and session; LRT by ML.

In [ ]:
def cond_rate_tables():
    rows = []
    for s in use:
        ses = TM[s]["ses"]
        u = TM[s]["units"].reset_index(drop=True)
        for ts in ["out", "in"]:
            B = ses.bigseg_rates(ts)
            for cond in ["Co", "Inco"]:
                tr = ses.cond[cond].values
                X = B[tr]
                n = np.sum(~np.isnan(X), axis=0)
                m = np.nanmean(X, axis=0)
                sd = np.nanstd(X, axis=0, ddof=1)
                with np.errstate(invalid="ignore", divide="ignore"):
                    z = np.where(n > 1, m / sd, np.nan)
                for j, segn in enumerate(["left", "stem", "right"]):
                    rows.append(pd.DataFrame(dict(session=s, subject=TM[s]["subject"], task=s.split("_")[1],
                                                  unit=np.arange(len(u)), unit_type=u.unit_type.values, ts=ts,
                                                  cond=cond, ms=segn, z=z[:, j], n=n[:, j])))
    T = pd.concat(rows, ignore_index=True)
    T["uid"] = T.session + "/" + T.unit.astype(str)
    keep = T[(T.ts == "out") & (T.cond == "Inco") & (T.ms == "stem") & (T.n > 5)].uid
    T = T[T.uid.isin(keep) & T.ms.isin(["left", "right"])]
    W = T.pivot_table(index=["uid", "subject", "session", "task", "unit_type", "ts", "ms"], columns="cond",
                      values="z").reset_index()
    W["dz"] = W.Co - W.Inco
    zo = set(W[(W.ts == "out") & (W.dz > 0)].uid)
    zi = set(W[(W.ts == "in") & (W.dz > 0)].uid)
    for D in (T, W):
        D["zo_p"] = D.uid.isin(zo)
        D["zi_p"] = D.uid.isin(zi)
    return T, W


def conditional_tests(T, W, ts, unit_set):
    t = T[T.ts == ts].dropna(subset=["z"])
    z = W[W.ts == ts].dropna(subset=["dz"]).rename(columns={"dz": "zz"})
    if unit_set == "cell":
        t, z = t[t.unit_type == "cell"], z[z.unit_type == "cell"]
        add = ""
    else:
        add = " + unit_type"
    cp = "zo_p" if ts == "in" else "zi_p"
    vc = {"task": "1+C(task)", "session": "0+C(session)"}
    f1, n1 = f"z ~ ms{add} + {cp} + cond", f"z ~ ms{add} + {cp}"
    m1 = tma.mixed(f1, t, vc)
    a = tma.lrt_models(f1, n1, t, vc)
    f2, n2 = f"zz ~ ms{add} + {cp}", f"zz ~ ms{add}"
    m2 = tma.mixed(f2, z, vc)
    b = tma.lrt_models(f2, n2, z, vc)
    return dict(test_on=ts, unit_set=unit_set, n_units=t.uid.nunique(), inco_coef=m1.params["cond[T.Inco]"],
                lrt_incorrect=a[0], p_incorrect=a[2], cond_coef=m2.params[f"{cp}[T.True]"],
                lrt_conditioning=b[0], p_conditioning=b[2], refit=a[3] or b[3])

In [ ]:
T7, W7 = cond_rate_tables()
print("units kept (more than 5 incorrect outbound trials with a stem rate):", T7.uid.nunique(),
      "; isolated:", T7[T7.unit_type == "cell"].uid.nunique())
print(W7.groupby(["unit_type", "ts", "zo_p"]).dz.agg(["count", "mean"]).round(3).to_string())
show([conditional_tests(T7, W7, "in", "cell"), conditional_tests(T7, W7, "out", "cell"),
      conditional_tests(T7, W7, "in", "all"), conditional_tests(T7, W7, "out", "all")])
print("stated direction: inco_coef > 0 (incorrect above correct) and cond_coef > 0 (units with correct above "
      "incorrect on the other run have a larger correct minus incorrect difference)")

**Result: reproduced.** On 510 isolated units, inbound runs conditioned on outbound: incorrect above correct (coefficient 0.146, LRT 4.04, p = 0.044) and the conditioning effect (0.280, LRT 10.5, p = 0.0012); outbound conditioned on inbound: incorrect above correct (0.391, LRT 25.4, p = 4.7e-7) and the conditioning effect (0.186, LRT 4.12, p = 0.042, after the Powell refit, as the default optimizer gave a negative statistic). With MUA the inbound conditioning effect is not significant (p = 0.17). Two of the four tests are close to 0.05 (0.044 and 0.042), so the result is fragile. The first run of this notebook printed LRT 4.12, p = 0.043 for the first test; this run prints 4.04, p = 0.044, and a re-run of the test in separate processes gave 4.04 each time.

## Zone encoders

Test-fold R2 per unit, fold and model (fitted in the loader). Per unit and model pair, the Mann-Whitney z of
the test-fold R2 of one model against the other, and the mean test R2 of each model
(`experiment_info.py, get_zone_encoder_comps`, with the pairs of `zone_encoder_comps_dict`). Units whose mean
test R2 is below -1 are left out (claims file; `paper_stats.py, _zone_encoder_scores`).

In [ ]:
ENC_COMPS = {"cue": [("Z+C", "Z", "fixed_v_none"), ("Z", "ZxC", "none_v_inter"), ("ZxC", "Z+C", "inter_v_fixed")],
             "rw": [("Zi+R", "Zi", "fixed_v_none"), ("Zi", "ZixR", "none_v_inter"), ("ZixR", "Zi+R", "inter_v_fixed")],
             "lag": [("Z50", "Z0", "pos_v_zero"), ("Z0", "Z-50", "zero_v_neg"), ("Z-50", "Z50", "neg_v_pos")]}
NAME = {"Z": "Z50", "Zi": "Zi"}


def enc_table(expt, run):
    key = {"rw": "enc_rw", "cue": "enc_cue", "lag": "enc_lag"}[expt]
    rows = []
    for s in use:
        e = TM[s][key]
        e = e[e.run == run]
        u = TM[s]["units"].reset_index(drop=True)
        P = e.pivot_table(index="fold", columns=["model", "unit"], values="r2")
        for test, null, comp in ENC_COMPS[expt]:
            a, b = P[NAME.get(test, test)], P[NAME.get(null, null)]
            for k in a.columns:
                rows.append(dict(session=s, subject=TM[s]["subject"], task=s.split("_")[1], unit=k,
                                 unit_type=u.unit_type[k], comp=comp, test_cond=test, null_cond=null,
                                 mean_test=a[k].mean(), mean_null=b[k].mean(),
                                 uz=tma.mannwhitney_z(a[k].values, b[k].values)))
    out = pd.DataFrame(rows)
    out.loc[out.mean_test < -1, ["mean_test", "uz"]] = np.nan
    return out


def enc_lmem(E, unit_set, score):
    d = E.dropna(subset=[score]).copy()
    if unit_set == "cell":
        d = d[d.unit_type == "cell"]
    d["score"] = d[score]
    vc = {"task": "1+C(task)", "session": "0+C(session)"}
    if unit_set == "all":
        vc["unit"] = "0+C(unit_type)"
    term = "comp" if score == "uz" else "test_cond"
    f = f"score ~ 0 + {term}"
    m = tma.mixed(f, d, vc)
    a = tma.lrt_models(f, "score ~ 1", d, vc)
    out = dict(unit_set=unit_set, reading=score, n_units=d.groupby(["session", "unit"]).ngroups, lrt=a[0], df=a[1], p=a[2],
               refit=a[3])
    out.update({k.replace(f"{term}[", "").rstrip("]"): v for k, v in m.fe_params.items()})
    out.update({"p_" + k.replace(f"{term}[", "").rstrip("]"): v for k, v in m.pvalues.items() if k in m.fe_params.index})
    if score == "mean_test":
        # Wald contrasts of one model against another on the REML fit (the fixed effects above test each model
        # against zero): difference of the two model means, z from their covariance, two-sided p.
        fe, cov = m.fe_params, m.cov_params()
        levels = [k.replace("test_cond[", "").rstrip("]") for k in fe.index]
        for a_ in levels:
            for b_ in levels:
                if a_ < b_:
                    ka, kb = f"test_cond[{a_}]", f"test_cond[{b_}]"
                    diff = fe[ka] - fe[kb]
                    se = np.sqrt(cov.loc[ka, ka] + cov.loc[kb, kb] - 2 * cov.loc[ka, kb])
                    out[f"{a_} vs {b_}"] = diff
                    out[f"p {a_} vs {b_}"] = 2 * stats.norm.sf(abs(diff / se))
    return out


def enc_section(expt, best, others, runs):
    rows = []
    means = []
    for run in runs:
        E = enc_table(expt, run)
        means.append(E[E.unit_type == "cell"].groupby("test_cond").mean_test.mean().rename(run))
        for us in ["cell", "all"]:
            for score in ["uz", "mean_test"]:
                rows.append(dict(run=run, **enc_lmem(E, us, score)))
        if run == runs[0]:
            print("mean test R2 per model (units kept):")
            print(E.groupby(["unit_type", "test_cond"]).mean_test.agg(["count", "mean", "median"]).round(4).to_string())
            print("units left out (mean test R2 below -1):", int(E.mean_test.isna().sum()), "of", len(E))
            print("fraction of units whose best model is", best, ":",
                  (E.pivot_table(index=["session", "unit"], columns="test_cond", values="mean_test").idxmax(axis=1) == best)
                  .mean().round(3))
    print("mean test R2 of isolated units per model, by run:")
    print(pd.DataFrame(means).round(4).to_string())
    out = pd.DataFrame(rows)
    pw = [c for c in out.columns if " vs " in c]
    print("R2 reading, Wald contrasts between models (difference of the LMEM means, two-sided p):")
    print(out[out.reading == "mean_test"][["run", "unit_set"] + pw].to_string(float_format=lambda x: f"{x:.4g}"))
    return out.drop(columns=pw)

## fig3d-cue-rate-remapping-model

> A cue rate-remapping encoder (zones plus cue gain) predicts single-unit firing better than the zone-only
> and the cue global-remapping models, with a mean R2 of 12%.

In [ ]:
RUNS = [f"5 folds, seed+{o}" for o in [0, 1000, 2000, 3000, 4000]] + ["10 folds"]
ENC3 = enc_section("cue", "Z+C", ["Z", "ZxC"], RUNS)
print(ENC3.to_string(float_format=lambda x: f"{x:.4g}"))
print("Z+C beats Z when fixed_v_none > 0 (uz) or Z+C > Z (mean_test); beats ZxC when inter_v_fixed < 0 or Z+C > ZxC")
print("expected mean Z+C test R2 0.12 +- 0.012")

**Result: reproduced.** On the Mann-Whitney reading with 5 folds, the comparison LRT is 1196 (df 2, p = 2.4e-260) for isolated units and Z+C beats Z (fixed_v_none 0.846) and ZxC (inter_v_fixed -1.016); with MUA 4361 (p < 1e-300) and the same signs. The mean test R2 of Z+C over isolated units is 0.123 (expected 0.12 +- 0.012), against 0.113 for Z and 0.109 for ZxC, and 82% of all units are best fitted by Z+C; five seeds give 0.1225 to 0.1236 and 10 folds 0.1229. On the second reading (R2 by model) the LRT is 6.0, p = 0.0499, with Z+C highest (p 0.0499 to 0.067 under the five seeds, 0.073 with 10 folds). The single comparisons are weaker (fixed_v_none p = 0.056, inter_v_fixed p = 0.022). The Wald contrasts on the R2 reading put Z+C above Z by 0.010 (p = 0.094) and above ZxC by 0.015 (p = 0.017) for isolated units (p 0.081 to 0.10 and 0.017 to 0.029 over the seeds and 10 folds), and with MUA above both (p = 0.0061 and 1.5e-5).

## fig5-reward-rate-remapping-model

> On inbound runs, a reward rate-remapping encoder fits most units better than the zone-only and reward
> global-remapping models.

In [ ]:
ENC5 = enc_section("rw", "Zi+R", ["Zi", "ZixR"], RUNS)
print(ENC5.to_string(float_format=lambda x: f"{x:.4g}"))

**Result: reproduced.** On the Mann-Whitney reading the LRT is 813 (p = 2.9e-177) for isolated units with Zi+R beating Zi (0.967) and ZixR (-0.812), and 2994 with MUA; on the R2 reading the LRT is 13.2, p = 0.0014, with Zi+R highest (mean 0.123 against 0.102 and 0.101); 77% of all units are best fitted by Zi+R. Five seeds and 10 folds give the same signs. The Wald contrasts on the R2 reading put Zi+R above Zi by 0.021 (p = 0.0018) and above ZixR by 0.022 (p = 0.0014) for isolated units, with p below 0.007 under every seed and with 10 folds.

## methods-lookahead-encoder-best

> Among zone encoders with retrospective, no, and prospective lags, the 1 s look-ahead encoder performed best.

In [ ]:
ENCL = enc_section("lag", "Z50", ["Z0", "Z-50"], RUNS)
print(ENCL.to_string(float_format=lambda x: f"{x:.4g}"))
print("Z50 is favoured over Z0 when pos_v_zero > 0 and over Z-50 when neg_v_pos < 0")

**Result: partially reproduced.** Z50 has the highest mean test R2 over isolated units (0.114, against 0.109 for Z0 and 0.099 for Z-50; the same order under all five seeds and with 10 folds), but neither comparison favours it at p < 0.05 (Z50 against Z0: 0.018, p = 0.98; Z-50 against Z50: -0.687, p = 0.36). With MUA included, Z0 has the higher raw mean among MUA (0.046 against 0.045) and Z50 against Z0 is -0.195 (p = 0.80). On the R2 reading, the Wald contrasts put Z50 above Z-50 by 0.015 (p = 0.011) but above Z0 by only 0.0045 (p = 0.44) for isolated units (p 0.011 to 0.022 and 0.39 to 0.44 over the seeds and 10 folds), and with MUA 0.0085 above Z-50 (p = 0.00066) and 0.0002 above Z0 (p = 0.93). So under both readings Z50 is not favoured over Z0 at p < 0.05.

## fig6cd-of-aggregate-model

> In open-field foraging, an aggregate model built from the speed, head-direction and position model
> predictions predicts single-unit firing on test data better than the individual models.

Models as in the paper's code for the published analysis (`experiment_info.py,
get_of_encoding_models_scores2`: speed, head direction, position and their aggregate, responses z-scored per
fold) with the parameters of the claims file and the code (see `tma.of_models`). Isolated units only, as
`paper_stats.py, _open_field_models` drops MUA (and so only isolated units are fitted); units with any R2 at or
below -1 are left out. Blocks of 20 s (Methods) and, as a variant, 30 s (the code's call). The session the code
marks invalid (`Li_OF_080718`) is left out. The folds are fixed blocks, so no seed is involved.

In [ ]:
OFP = tma.of_params()
POS_PCA = tma._position_pca(OFP)
print("position features: PCA components", POS_PCA[1].n_components_, "for", POS_PCA[0].shape[0], "bins")
of_rows = [row for _, row in files[~files.tm].iterrows() if row.n_cell > 0 and row.session not in tma.INVALID_SESSIONS]


def run_of(row):
    a = assets[assets.path == row.path].iloc[0]
    f = tma.open_asset(a.asset_id)
    ut = meta_by_session[row.session]["units"].unit_type.values
    cells = np.where(ut == "cell")[0]
    fr = f["acquisition/firing_rates/data"][:, cells].T.astype(float)
    tr = f["acquisition/track_data/data"][:]
    f.close()
    out = []
    for secs in [20.0, 30.0]:
        r = tma.of_models(fr, tr, OFP, POS_PCA, secs_per_split=secs)
        if r is not None:
            r["blocks_s"] = secs
            out.append(r)
    rng_ = dict(x=(tr[:, 0].min(), tr[:, 0].max()), y=(tr[:, 1].min(), tr[:, 1].max()), sp_max=tr[:, 3].max(),
                n=tr.shape[0])
    if not out:
        return row.session, None, rng_
    r = pd.concat(out, ignore_index=True)
    r["session"], r["subject"] = row.session, row.subject
    return row.session, r, rng_


t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    of_res = list(ex.map(run_of, of_rows))
OF = pd.concat([r for _, r, _ in of_res if r is not None], ignore_index=True)
OF["uid"] = OF.session + "/" + OF.unit.astype(str)
print(f"open-field sessions with isolated units: {len(of_rows)}; fitted: {OF.session.nunique()}; "
      f"isolated units {OF.uid.nunique()}; {time.time() - t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")
print("sessions too short for 5 blocks:", [s for s, r, _ in of_res if r is None])
R_of = pd.DataFrame({s: g for s, _, g in of_res}).T
print("open-field track ranges (cm, cm/s): x", R_of.x.map(lambda v: v[0]).min().round(1), "to",
      R_of.x.map(lambda v: v[1]).max().round(1), "; y", R_of.y.map(lambda v: v[0]).min().round(1), "to",
      R_of.y.map(lambda v: v[1]).max().round(1), "; max speed", R_of.sp_max.max().round(1),
      "; the description says mm, the values are in cm")


def of_lmem(d, score):
    """paper_stats.py, _open_field_models, joint_fit: value ~ 0 + model against value ~ 1, groups subject,
    variance components session and unit; test-fold values of isolated units."""
    d = d.rename(columns={score: "value"}).dropna(subset=["value"])
    vc = {"session": "0+C(session)", "unit_id": "0+C(uid)"}
    m = tma.mixed("value ~ 0 + model", d, vc)
    a = tma.lrt_models("value ~ 0 + model", "value ~ 1", d, vc)
    out = dict(score=score, n_units=d.uid.nunique(), lrt=a[0], df=a[1], p=a[2], refit=a[3])
    out.update({k.replace("model[", "").rstrip("]"): v for k, v in m.fe_params.items()})
    return out

In [ ]:
rows = []
for secs in [20.0, 30.0]:
    d = OF[OF.blocks_s == secs]
    bad = d[d.r2 <= -1].uid.unique()
    d = d[~d.uid.isin(bad)]
    print(f"blocks {secs:.0f} s: units left out for an R2 <= -1: {len(bad)}")
    print(d.groupby("model")[["r2", "map_r"]].agg(["mean", "median"]).round(4).to_string())
    best = d.groupby(["uid", "model"]).r2.mean().unstack().idxmax(axis=1).value_counts()
    print("units by best model (mean test R2):", best.to_dict())
    for score in ["r2", "map_r"]:
        rows.append(dict(blocks_s=secs, **of_lmem(d, score)))
show(rows)
print("criterion: both LRTs p < 0.05 and the aggregate model highest in test R2 and r_p")
print("by subject, mean test R2 per model (20 s blocks):")
print(OF[(OF.blocks_s == 20.0)].groupby(["subject", "model"]).r2.mean().unstack().round(4).to_string())

**Result: reproduced.** Over 664 isolated units in 103 open-field sessions (one unit left out for an R2 at or below -1), with 20 s blocks, the aggregate model has the highest mean test R2 (0.096, against 0.043 for position, 0.027 for head direction and 0.007 for speed) and the highest r_p (0.406, against 0.286, 0.276 and 0.135), and is the best model for 452 units. The model effect has LRT 2450 (df 3) on R2 and 3664 on r_p, both with p below the floating-point limit (printed as 0); the paper reports chi2(4) = 567.15 and 878.84. With 30 s blocks the order is the same (0.093 and 0.404 for the aggregate). By subject the aggregate has the highest mean R2 in four of five; in Ca all models are near zero (aggregate -0.008, position 0.003).

## Figure 7 on the paper's own matches

The matching of fig7b-matched-units cannot be rerun: the files hold no waveforms. The paper's matches are in its
repository's `metadata/unit_table.csv` (at the cited commit, an input), with the unit names (`cl_name_T3`,
`cl_name_OF`), uuids and the open-field cluster of each match (`of_cluster_id`). The first cell checks those names
and uuids against the dandiset's unit tables. fig7g-hd-cluster-remaps-more and fig7h-hd-cluster-encoder-scores take
their clusters from that table (`data: with_labels`); their verified outcomes stay blocked in this round, and the
tests below are the analysis behind the change of outcome proposed for each in the attempt record, for a third
verification round. fig7de-hd-speed-tradeoff and fig7f-tuning-metrics-match are not tested here.

In [ ]:
UT = pd.read_csv(TMA + "unit_table.csv", index_col=0)
print(f"the paper's match table: {len(UT)} rows (Tree-Maze unit, open-field unit), {UT.match_cl_id.nunique()} matched "
      f"units, {UT.session_T3.nunique()} Tree-Maze and {UT.session_OF.nunique()} open-field sessions")
print("matched units per cluster:", UT.groupby("of_cluster_id").match_cl_id.nunique().to_dict(),
      "; rows per cluster:", UT.of_cluster_id.value_counts().sort_index().to_dict(),
      "(the paper: cluster 0 n = 98, cluster 2 n = 59)")
U = pd.concat([r["units"].reset_index(drop=True).assign(session=r["session"], unit=np.arange(len(r["units"])))
               for r in meta if len(r["units"])], ignore_index=True)
print("unit names duplicated in the dandiset:", int(U.cl_name.duplicated().sum()))
by_name = U.set_index("cl_name")
for side in ["T3", "OF"]:
    nm = UT[f"cl_name_{side}"]
    found = nm.isin(by_name.index)
    same_uuid = found & (by_name.uuid.reindex(nm).values == UT[f"uuid_{side}"].values)
    same_ses = found & (by_name.session.reindex(nm).values == UT[f"session_{side}"].values)
    print(f"{side}: rows whose unit name is in the dandiset {int(found.sum())} of {len(UT)}; of them with the same uuid "
          f"{int(same_uuid.sum())} and the same session {int(same_ses.sum())}; sessions not in the dandiset: "
          f"{sorted(UT.loc[~found, f'session_{side}'].unique())}")
M = UT.join(by_name[["session", "unit", "unit_type"]].add_suffix("_tm"), on="cl_name_T3")
M["of_in_ds"] = M.cl_name_OF.isin(by_name.index)
M = M[M.session_tm.notna() & M.of_in_ds].copy()
M["in_use"] = M.session_tm.isin(use)
M["cluster"] = pd.Categorical(M.of_cluster_id)
print(f"rows with both units in the dandiset: {len(M)}; with the Tree-Maze session among the {len(use)} of the neural "
      f"analyses: {int(M.in_use.sum())} ({M[M.in_use].match_cl_id.nunique()} matched units, "
      f"{M[M.in_use].session_tm.nunique()} Tree-Maze sessions, {M[M.in_use].session_OF.nunique()} open-field sessions)")
print("Tree-Maze sessions of the table left out of the neural analyses:", sorted(M.loc[~M.in_use, "session_tm"].unique()))
print("unit types of the matched Tree-Maze units:", M[M.in_use].drop_duplicates(["session_tm", "unit_tm"]).unit_type_tm
      .value_counts().to_dict())
M = M[M.in_use].copy()
M["unit_tm"] = M.unit_tm.astype(int)
print("rows per cluster used:", M.cluster.value_counts().sort_index().to_dict(),
      "; matched units:", M.groupby("cluster", observed=True).match_cl_id.nunique().to_dict())


VC7 = {"session_TM": "0+C(session_T3)", "session_OF": "0+C(session_OF)", "match_id": "0+C(match_cl_id)"}


def fig7_table(cols):
    """paper_stats.py, _of_cluster_TM_comps: rows of the match table with every score present (dropna), melted."""
    d = M[["subject", "match_cl_id", "session_T3", "session_OF", "cluster"] + list(cols)].dropna()
    return d.melt(id_vars=["subject", "match_cl_id", "session_T3", "session_OF", "cluster"], value_vars=list(cols),
                  var_name="stype", value_name="score")

## fig7g-hd-cluster-remaps-more

> On matched units, cue and reward remapping remain significant, and units of the head-direction cluster 0 remap
> more than those of the other clusters.

The scores are the cue and reward remapping scores of the matched Tree-Maze units computed above (under the seeds of
fig2d-cue-remapping and fig4d-reward-remapping and four more). The model is that of `paper_stats.py,
_of_cluster_TM_comps` (score group `tm_remap`): score ~ 1 + remap type * cluster against score ~ 1 + remap type +
cluster (interaction), and that against score ~ 1 + remap type (main effect), groups subject, variance components
Tree-Maze session, open-field session and match id, LRT by ML; the cluster is categorical, as in the code's notebook
(`Notebooks/TM/paper_stats.ipynb`, cell 14). The intercepts are judged as in fig2d-cue-remapping, on the matched
Tree-Maze units.

In [ ]:
rows, means = [], []
for off in OFFS:
    M["cue"] = M.set_index(["session_tm", "unit_tm"]).index.map(SC.set_index(["session", "unit"])[f"cue_s{off}"])
    M["rw"] = M.set_index(["session_tm", "unit_tm"]).index.map(SC.set_index(["session", "unit"])[f"rw_s{off}"])
    d = fig7_table(["cue", "rw"])
    full, add, base = "score ~ 1 + stype * cluster", "score ~ 1 + stype + cluster", "score ~ 1 + stype"
    a = tma.lrt_models(full, add, d, VC7)
    b = tma.lrt_models(add, base, d, VC7)
    mu = d.groupby(["stype", "cluster"], observed=True).score.mean().unstack("cluster")
    r = dict(seed=f"claim+{off}", rows=len(d) // 2, matched_units=d.match_cl_id.nunique(), lrt_inter=a[0], df_inter=a[1],
             p_inter=a[2], lrt_main=b[0], df_main=b[1], p_main=b[2], refit=a[3] or b[3])
    uniq = M.drop_duplicates(["session_tm", "unit_tm"]).rename(columns={"session_tm": "session"})
    uniq["task"] = uniq.session.str.split("_").str[1]
    uniq["unit_type"] = uniq.unit_type_tm
    for st in ["cue", "rw"]:
        e = remap_lmem(uniq, st, "all" if uniq.unit_type.nunique() > 1 else "cell")
        r.update({f"{st}_mean": e["mean"], f"{st}_ci_high": e["ci_high"], f"{st}_p": e["p"], f"{st}_n": e["n"]})
        # variants: intercept with the random structure of the fig7g model, on the rows of the match table; and a
        # Wilcoxon signed-rank test of the matched units' scores against zero
        m0 = tma.mixed("score ~ 1", d[d.stype == st], VC7)
        r[f"{st}_rows_mean"], r[f"{st}_rows_ci_high"] = m0.params["Intercept"], m0.conf_int().loc["Intercept"].values[1]
        r[f"{st}_wilcoxon_p"] = stats.wilcoxon(uniq[st].dropna()).pvalue
        for k, v in mu.loc[st].items():
            r[f"{st}_mean_c{k}"] = v
    rows.append(r)
F7G = pd.DataFrame(rows)
print(F7G.T.to_string(float_format=lambda x: f"{x:.4g}"))
for _, r in F7G.iterrows():
    most = {st: min([0, 1, 2], key=lambda k: r[f"{st}_mean_c{k}"]) for st in ["cue", "rw"]}
    print(f"{r.seed}: intercept CIs below zero, cue {r.cue_ci_high < 0}, reward {r.rw_ci_high < 0}; interaction p < 0.05 "
          f"{r.p_inter < 0.05}; cluster with the most negative mean, cue {most['cue']}, reward {most['rw']}")
print("criterion: both intercept CIs below zero, interaction p < 0.05, cluster 0 most negative for cue or reward; "
      "the paper gives chi2(2) = 6.87, p = 0.032 (interaction) and chi2(2) = 1.5, p = 0.47 (main effect)")
by_subj = fig7_table(["cue", "rw"]).groupby(["subject", "stype", "cluster"], observed=True).score.agg(["count", "mean"])
print("\nby subject (last seed), mean score per cluster:")
print(by_subj.unstack("cluster").round(3).to_string())

**Result: blocked in the verified record; the proposed change is to partially reproduced.** The paper's match table names the dandiset's units: all 274 of its 306 rows whose Tree-Maze session is in the dandiset have the same unit name, uuid and session there, as do all 306 open-field units. 255 rows (185 matched units, all isolated) have their Tree-Maze session among the 123 of the neural analyses and 253 have both scores. Cluster 0 has the most negative mean cue (-0.717 against -0.624 and -0.626) and reward (-1.289 against -1.024 and -0.859) remapping score under every seed, but the interaction of remap type and cluster has LRT 5.74, p = 0.057 (paper 6.87, p = 0.032), and 0.057 to 0.19 under the five seeds; the main effect is not significant (p = 0.85; paper p = 0.47). Judged as in fig2d-cue-remapping, the reward intercept is -0.877 with a CI up to 0.093 (p = 0.076), not below zero under any seed, and the cue intercept's CI could not be computed under the claim's seed (below zero under the other four, up to -0.195). With the random structure of the fig7g model both intercepts have CIs below zero (cue up to -0.439, reward up to -0.221), and Wilcoxon tests of the matched units' scores give p below 1e-21. By subject (scores of the seed plus 4000), Al and Ca have one and two matched rows, and five of their six cluster means of cue and reward scores are above zero.

## fig7h-hd-cluster-encoder-scores

> Tree-Maze rate and global remapping encoder scores are higher for the head-direction cluster 0.

The scores are the mean test R2 of the rate (Z+C, Zi+R) and global (ZxC, ZixR) encoders of the matched Tree-Maze
units, from the encoders fitted above for fig3d-cue-rate-remapping-model and fig5-reward-rate-remapping-model, with
values of absolute value above 1 left out (`experiment_info.py, get_tm_scores`). The model is `_of_cluster_TM_comps`
with score group `tm_enc_r2`, test `main`: score ~ 1 + rate + cue + cluster against score ~ 1 + rate + cue, same
random structure as fig7g.

In [ ]:
def enc_unit_means(key, run):
    out = []
    for s in sorted(M.session_tm.unique()):
        e = TM[s][key]
        g = e[e.run == run].groupby(["unit", "model"]).r2.mean().unstack("model")
        out.append(g.assign(session=s).reset_index())
    return pd.concat(out, ignore_index=True).set_index(["session", "unit"])


rows = []
for run_label in RUNS:
    ec, er = enc_unit_means("enc_cue", run_label), enc_unit_means("enc_rw", run_label)
    k = M.set_index(["session_tm", "unit_tm"]).index
    M["rate_cue"], M["global_cue"] = k.map(ec["Z+C"]), k.map(ec["ZxC"])
    M["rate_rw"], M["global_rw"] = k.map(er["Zi+R"]), k.map(er["ZixR"])
    for c in ["rate_cue", "global_cue", "rate_rw", "global_rw"]:
        M.loc[M[c].abs() > 1, c] = np.nan
    d = fig7_table(["rate_cue", "global_cue", "rate_rw", "global_rw"])
    d["rate"] = d.stype.str.startswith("rate")
    d["cue"] = d.stype.str.endswith("cue")
    a = tma.lrt_models("score ~ 1 + rate + cue + cluster", "score ~ 1 + rate + cue", d, VC7)
    m = tma.mixed("score ~ 1 + rate + cue + cluster", d, VC7)
    mu = d.groupby("cluster", observed=True).score.mean()
    r = dict(run=run_label, rows=len(d) // 4, matched_units=d.match_cl_id.nunique(), lrt=a[0], df=a[1], p=a[2],
             refit=a[3], coef_c1=m.params["cluster[T.1]"], coef_c2=m.params["cluster[T.2]"])
    r.update({f"mean_c{c}": v for c, v in mu.items()})
    r.update({f"{st}_c{c}": v for (st, c), v in d.groupby(["stype", "cluster"], observed=True).score.mean().items()})
    rows.append(r)
F7H = pd.DataFrame(rows)
print(F7H.T.to_string(float_format=lambda x: f"{x:.4g}"))
for _, r in F7H.iterrows():
    best = max([0, 1, 2], key=lambda c: r[f"mean_c{c}"])
    print(f"{r.run}: LRT p < 0.05 {r.p < 0.05}; cluster with the highest mean score {best}")
print("criterion: LRT p < 0.05 and cluster 0 with the highest mean score; the paper gives chi2(2) = 13.89, p = 0.00096")

**Result: blocked in the verified record; the proposed change is to partially reproduced.** Over 255 rows (185 matched units), cluster 0 has the highest mean encoder score (0.178 against 0.086 and 0.063), and the highest for each of the four scores, under every seed and with 10 folds, but the cluster effect has LRT 4.05, df 2, p = 0.13 (0.12 to 0.19 over the runs), against the paper's chi2(2) = 13.89, p = 0.00096.

## Claims not tested

- fig1-training-to-criterion needs the training sessions before surgery. The dandiset holds only sessions
  with recordings (all its Tree-Maze files have units and a trial table, listed above); the code reads the
  training sessions from a spreadsheet (`experiment_info.py, get_all_behav_perf`, `all_behavior_perf`) that
  is not in the repository's metadata folder. Blocked, data_missing.
- fig7b-matched-units matches units across sessions by their waveforms. The files hold binned firing
  rates only (listed above: no spike times, no waveforms, no electrode table). Blocked, data_missing.
- fig7de-hd-speed-tradeoff and fig7f-tuning-metrics-match could be tested on the paper's own matches, whose
  unit names are the dandiset's (above), by clustering the coefficients of the code's open-field models of
  the matched units and computing their tuning scores; this attempt did not run them. Blocked, other.
- fig7g-hd-cluster-remaps-more and fig7h-hd-cluster-encoder-scores are blocked, other, in the verified record;
  the tests above, on the paper's clusters, are the analysis behind a proposed change of their outcomes.
- fig3e-zone-decoder-cue, fig3f-decision-decoder and fig3g-decoder-tracks-behaviour need the paper's
  population decoders (`tree_maze_functions.py, ZoneDecoder`), which this attempt did not run. The data
  they need are in the dandiset. Blocked, other.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

| claim | outcome |
|---|---|
| fig1d-task-performance | partially reproduced (accuracy matches; trials per session 152, not 110) |
| fig1-switch-performance | partially reproduced (accuracy and similarity hold; 69 Switch trials, not 54) |
| fig1-training-to-criterion | blocked, data_missing |
| fig2c-cue-error-coding | reproduced |
| fig2d-cue-remapping | reproduced |
| fig2e-unit-remap-behaviour | reproduced |
| fig2f-session-remap-behaviour | partially reproduced (slope -0.385, outside -0.22 +- 0.11) |
| fig2g-population-remap-behaviour | reproduced |
| fig2h-subject-remap-behaviour | reproduced (all five subjects negative) |
| fig2-remap-behaviour-controls | reproduced |
| fig3d-cue-rate-remapping-model | reproduced |
| fig3e-zone-decoder-cue | blocked, other (decoders not run in this attempt) |
| fig3f-decision-decoder | blocked, other |
| fig3g-decoder-tracks-behaviour | blocked, other |
| fig4c-reward-error-coding | reproduced |
| fig4d-reward-remapping | reproduced |
| fig4ef-reward-remap-behaviour | partially reproduced (two of four values outside tolerance) |
| fig4gh-reward-remap-population-subject | reproduced |
| fig4-reward-remap-behaviour-controls | reproduced |
| fig5-reward-rate-remapping-model | reproduced |
| fig4s5-cue-reward-remap-correlated | reproduced |
| fig4s6-incorrect-coding-overlap | reproduced, seed sensitive |
| fig4s7-error-signal-persists | reproduced |
| fig6cd-of-aggregate-model | reproduced |
| fig7b-matched-units | blocked, data_missing |
| fig7de-hd-speed-tradeoff | blocked, other (not tested; can be tested on the paper's matches) |
| fig7f-tuning-metrics-match | blocked, other (not tested; can be tested on the paper's matches) |
| fig7g-hd-cluster-remaps-more | blocked, other; proposed partially reproduced (cluster 0 most negative; interaction p = 0.057; reward CI not below zero) |
| fig7h-hd-cluster-encoder-scores | blocked, other; proposed partially reproduced (cluster 0 highest; LRT p = 0.13) |
| methods-lookahead-encoder-best | partially reproduced (best on mean R2, comparisons not significant) |

How the analysis differs from the paper's:

- The analyses are ports of the paper's code at the cited commit, run on the dandiset's binned rates and
  tracking, which are what the code itself writes to the files. Random draws use the claims' seeds, not the
  code's seeds (0 for resampling, 42 for folds); the code's seed 0 gives the same remapping results. The
  control scores were computed under their claims' seeds only.
- Sessions: 124 Tree-Maze sessions with at least one unit and 50 trials; the neural analyses use 123, leaving
  out `Cl_T3g_121418_0` as the code does. The dandiset lacks 12 Tree-Maze sessions with units that the
  repository's session table lists (224 units, 65 of them isolated), so unit counts are below the paper's
  (510 isolated units against 565 in Figure 4c).
- Encoders: trials without a correct flag are left out of the encoder folds (the code leaves out only the
  first such trial). When an ML fit gave a negative LRT, both models were refitted with Powell's method
  (one test, in fig4s7).
- Open field: only isolated units are fitted, as the paper's statistics use only them.
- The decoders (Figure 3e to 3g) were not run, and the clustering of Figure 7d and 7e and the tuning scores of
  Figure 7f were not run. Figure 7g and 7h were tested on the paper's own matches and clusters
  (`metadata/unit_table.csv`), whose unit names, uuids and sessions agree with the dandiset's.
- Switch accuracy follows the code (all Switch trials in the denominator); the variant over Switch trials
  with a correct flag is shown.

What was found about the dataset:

- The files hold smoothed firing rates in 20 ms bins, labelled "uv", and no spike times, waveforms or
  electrode tables; unit matching across sessions (Figure 7) cannot be done from them.
- `track_data` in Tree-Maze files is described as "columns=x,y,hd" but holds t, x, y and head angle (radians),
  starts one sample after the rates, and its `rate` attribute is 0.02, a period. Open-field `track_data` is
  labelled mm but holds cm. No table column has a description; the meaning of the trial markers (t0, tD, tE,
  tE_1, tE_2, tR) is only in the code. Session start times are in years 0018 and 0019.
- The checks support the description otherwise: markers fall in the expected zones (medians over sessions:
  Home at the trial start in 100% of trials, D at the decision marker in 99.5%, the goal well matching the
  recorded decision in 98.3%, against 42.6% with decisions shifted by one trial), and zone maps agree best
  with rates unshifted against tracking in 114 of 124 sessions.
- The draft is the only version; there is no published version to pin.